# Knowledge Graph Schema Design

**Project:** KG-Based RecSys (MovieLens 100K + TMDB)  
**Task:** T-08 — Design the Knowledge Graph schema  
**Deliverable:** Documented node labels, relationship types, properties, constraints, and design decisions for the Neo4j graph that powers the recommendation engine.

This notebook is a **design document**, not an implementation script. The actual loading of data into Neo4j happens in T-10 (`04_graph_load.ipynb`). The Cypher shown here is documentation — copy/paste into Neo4j Browser if needed, but the notebook itself does not connect to the database.

**Companion artifact:** [`docs/03_schema_diagram.md`](../docs/03_schema_diagram.md) — Mermaid schema diagram (renders on GitHub and in VS Code markdown preview).

---

## Table of Contents

1. [Purpose & Scope](#1-purpose--scope)
2. [Competency Questions](#2-competency-questions)
3. [Schema Overview](#3-schema-overview)
4. [Node Specifications](#4-node-specifications)
5. [Relationship Specifications](#5-relationship-specifications)
6. [Constraints & Indexes](#6-constraints--indexes)
7. [Query Conventions](#7-query-conventions)
8. [Design Decisions Log](#8-design-decisions-log)
9. [Data Source Mapping](#9-data-source-mapping)
10. [Open Questions / Future Work](#10-open-questions--future-work)

---

## 1. Purpose & Scope

### What this schema is for

This schema underpins a teaching-grade recommendation system built on MovieLens 100K + TMDB. It must support:

- **3 Streamlit screens** — User Profile, Recommendations, Explainability Explorer
- **4 RecSys strategies** — Genre affinity, Director/Actor affinity, Collaborative filtering via shared neighborhood, Composite multi-signal
- **1 cold-start strategy** — recommendations for a user with 0 or 1 ratings via demographics + popularity
- **Explainability** — every recommendation must be backed by a traversable path in the graph (e.g. `User → rated → Movie → directed_by → Director → directed → recommended Movie`)

### Audience

This document is read by three roles, each with a different "aha":

- **Data Analysts** — see how a question like *"top 5 genres for users in occupation X"* becomes a 2-hop traversal instead of a SQL join + group-by.
- **ML Engineers** — see how the same graph powers content-based, collaborative, and hybrid recommenders without a separate feature pipeline.
- **Data Engineers** — see how 100K ratings + ~1.7K movies + ~10K nodes fit into Neo4j AuraDB Free, with constraints/indexes that keep load deterministic.

### Design criteria

Every decision is judged against three principles, in this order:

1. **Does a competency question require this?** If no CQ asks for it, it does not go in the graph.
2. **Is it cheaper to compute live than to materialize?** Derived edges (similarity, collaboration) are computed at query time, not stored.
3. **Does it fit AuraDB Free?** Hard ceiling of ~200k nodes/relationships. The schema is designed to land at ~10k nodes + ~110k relationships.

### Out of scope

- Production scaling (sharding, replicas, write throughput)
- A formal OWL/RDF ontology — this is a labeled property graph, not a description-logic ontology
- Multi-domain (only movies; no books, music, etc.)
- Streaming / real-time updates — the graph is loaded once from CSV + TMDB snapshots
- PII handling — MovieLens 100K is already anonymized (only `userId`, age, occupation, gender)

---

## 2. Competency Questions

Competency Questions (CQs) are the technique introduced by Gruninger & Fox (1995): write down, in plain language, every question the graph must answer **before** designing it. Each CQ then justifies at least one node label, relationship type, or property. If a schema element is not exercised by any CQ, it does not belong in the graph.

Our 19 CQs map to the 3 Streamlit screens plus a free-form "Graph Explorer" view that ships with the app.

### Area 1 — User Profile screen

The *"who is this user"* view. Shown when a user is first selected.

| ID | Question | Schema element justified |
|---|---|---|
| CQ-1.1 | What movies has user U rated, and with what scores? | `(:User)-[:RATED {rating, timestamp}]->(:Movie)` |
| CQ-1.2 | What is user U's demographic profile? (age, occupation, gender) | `User.age`, `(:User)-[:HAS_OCCUPATION]->(:Occupation)`, `(:User)-[:HAS_GENDER]->(:Gender)` |
| CQ-1.3 | What are user U's top N favorite genres, ranked by average rating? | `(:Movie)-[:IN_GENRE]->(:Genre)` + `RATED.rating` |
| CQ-1.4 | What directors and actors does user U rate highest on average? | `:DIRECTED_BY`, `:FEATURES_ACTOR` + `RATED.rating` |
| CQ-1.5 | What are the top genres among users sharing user U's occupation? | `:HAS_OCCUPATION` joining `User → Movie → Genre` |

### Area 2 — Recommendations screen

The four-strategy RecSys panel.

| ID | Question | Schema element justified |
|---|---|---|
| CQ-2.1 | Which movies in user U's favorite genre(s) has U not yet rated? | `:IN_GENRE` + `NOT EXISTS` guard on `:RATED` |
| CQ-2.2 | Which movies share a director or actor with U's top-rated movies, that U has not seen? | `:DIRECTED_BY`, `:FEATURES_ACTOR` — 3-hop traversal |
| CQ-2.3 | Which movies are highly rated by users whose taste overlaps with U's (≥ 3 shared movies rated 4+)? | `:RATED` — 4-hop collaborative pattern |
| CQ-2.4 | For a cold-start user (0–1 ratings), what is popular among users with the same demographics? | `:HAS_OCCUPATION`, `:HAS_GENDER`, `User.age` + popularity over `:RATED` |
| CQ-2.5 | What is the explainability path that justifies each recommendation? | Variable-length path `*1..4` returning the full chain |

### Area 3 — Explainability paths

The *"why was this recommended?"* detail view, shown when the user clicks a recommendation.

| ID | Question | Schema element justified |
|---|---|---|
| CQ-3.1 | What is the shortest path between user U and movie M in the graph? | `shortestPath` over all relationship types |
| CQ-3.2 | What are the top 3 distinct paths (length ≤ 4) connecting U and M? | Variable-length `*1..4`, capped at 3 results |
| CQ-3.3 | Which intermediate node (director, actor, genre, peer user) is most often on the path? | Path-node aggregation |
| CQ-3.4 | Can the same recommendation be reached via multiple independent paths? | Path enumeration |

> **Convention:** `:Gender` is **excluded** from explanation paths. With only 2 nodes (`M`, `F`), it is a supernode and any path through it is uninformative ("recommended because you and 470 other users are both female"). The exclusion is implemented in the query, not the schema — Gender is still loaded as a node so CQ-1.2 and CQ-2.4 can use it.

### Area 4 — Graph Explorer screen

A free-form view for the teaching/demo audience to traverse the graph.

| ID | Question | Schema element justified |
|---|---|---|
| CQ-4.1 | Show the local neighborhood of any movie/user/director/actor (≤ 2 hops, ≤ 50 nodes). | All node labels + all relationship types |
| CQ-4.2 | Which actors most frequently appear alongside a given actor? | `:FEATURES_ACTOR` 2-hop via shared `:Movie` |
| CQ-4.3 | Which directors have the highest average rating across all their movies? | `:DIRECTED_BY` + `RATED.rating` aggregation |
| CQ-4.4 | Which genre combinations appear most often together on the same movie? | `:IN_GENRE` co-occurrence |
| CQ-4.5 | Find a movie by partial title match. | Full-text index on `Movie.title` |

### CQ → Schema element coverage

Sanity check that every node label and relationship type is exercised by at least one CQ.

| Schema element | CQs that exercise it |
|---|---|
| `:User` | 1.1, 1.2, 1.3, 1.4, 1.5, 2.1–2.5, 3.1–3.4, 4.1 |
| `:Movie` | 1.1, 1.3, 1.4, 2.1–2.5, 3.1–3.4, 4.1, 4.3–4.5 |
| `:Genre` | 1.3, 1.5, 2.1, 4.1, 4.4 |
| `:Director` | 1.4, 2.2, 4.1, 4.3 |
| `:Actor` | 1.4, 2.2, 4.1, 4.2 |
| `:Occupation` | 1.2, 1.5, 2.4, 4.1 |
| `:Gender` | 1.2, 2.4, 4.1 |
| `:RATED` | 1.1, 1.3, 1.4, 2.1–2.5, 3.1–3.4, 4.1, 4.3 |
| `:IN_GENRE` | 1.3, 1.5, 2.1, 4.1, 4.4 |
| `:DIRECTED_BY` | 1.4, 2.2, 4.1, 4.3 |
| `:FEATURES_ACTOR` | 1.4, 2.2, 4.1, 4.2 |
| `:HAS_OCCUPATION` | 1.2, 1.5, 2.4, 4.1 |
| `:HAS_GENDER` | 1.2, 2.4, 4.1 |

Every label and every relationship is covered by ≥ 1 CQ. No orphan schema elements.

---

## 3. Schema Overview

The graph has **7 node labels** and **6 relationship types**, organized in three tiers.

### Node tiers

| Tier | Node labels | Identifier | Notes |
|---|---|---|---|
| **Entity** | `:User`, `:Movie` | numeric ID from source | Primary entities; the two "anchors" of every traversal |
| **Person** | `:Director`, `:Actor` | `name` | Real people from TMDB |
| **Categorical** | `:Genre`, `:Occupation`, `:Gender` | `name` | Low-cardinality reference nodes |

### ASCII overview

Arrows show **stored direction**, but most queries traverse the graph **undirected** (see §7 Query Conventions).

```text
                  [Occupation]              [Gender]
                       ^                       ^
                       | HAS_OCCUPATION        | HAS_GENDER
                       |                       |
                       +-----------+-----------+
                                   |
                       [User {userId, age}]
                                   |
                                   | RATED {rating, timestamp}
                                   v
                       [Movie {movieId, title, year, tmdbId}]
                                   |
                       +-----------+-------------+
                       |           |             |
                       | IN_GENRE  | DIRECTED_BY | FEATURES_ACTOR {billing_order}
                       v           v             v
                   [Genre]    [Director]      [Actor]
```

For a color-coded diagram, see [`docs/03_schema_diagram.md`](../docs/03_schema_diagram.md).

### Inventory

- **Node labels (7):** `:User`, `:Movie`, `:Genre`, `:Director`, `:Actor`, `:Occupation`, `:Gender`
- **Relationship types (6):** `:RATED`, `:IN_GENRE`, `:DIRECTED_BY`, `:FEATURES_ACTOR`, `:HAS_OCCUPATION`, `:HAS_GENDER`

### Expected size at load

| Group | Count | Source |
|---|---:|---|
| `:User` nodes | 943 | MovieLens `u.user` |
| `:Movie` nodes | ~1,680 | MovieLens `u.item` (after TMDB match) |
| `:Genre` nodes | 19 | MovieLens genre vocabulary |
| `:Director` nodes | ~1,500 | TMDB credits |
| `:Actor` nodes | ~6,000 | TMDB credits, top 10 by billing per movie |
| `:Occupation` nodes | 21 | MovieLens occupation vocabulary |
| `:Gender` nodes | 2 | MovieLens `M` / `F` |
| **Total nodes** | **~10,200** | |
| `:RATED` edges | 100,000 | `u.data` |
| `:IN_GENRE` edges | ~3,000 | `u.item` (avg ~1.8 genres/movie) |
| `:DIRECTED_BY` edges | ~1,800 | TMDB credits |
| `:FEATURES_ACTOR` edges | ~15,000 | TMDB credits (top 10/movie) |
| `:HAS_OCCUPATION` edges | 943 | `u.user` |
| `:HAS_GENDER` edges | 943 | `u.user` |
| **Total edges** | **~121,700** | |

Comfortably under the AuraDB Free ~200k node/relationship ceiling.

---

## 4. Node Specifications

Per node label: properties, types, key, source, expected cardinality, and design notes.

### 4.1 `:User`

Primary entity for the rating/recommendation side of the graph.

| Property | Type | Key? | Source | Description |
|---|---|---|---|---|
| `userId` | `int` | ✅ unique | `u.user.user_id` | MovieLens user ID, 1–943 |
| `age` | `int` |   | `u.user.age` | Raw age in years (7–73 in the dataset) |

- **Expected cardinality:** 943 nodes
- **Notes:** Age is stored as a raw integer, not bucketed — bucketing is deferred to §10 so cold-start logic can experiment with window sizes. Occupation and gender are *not* stored as `:User` properties — they are first-class nodes so CQ-1.5 and CQ-2.4 can traverse them.

### 4.2 `:Movie`

Primary entity for everything being recommended.

| Property | Type | Key? | Source | Description |
|---|---|---|---|---|
| `movieId` | `int` | ✅ unique | `u.item.movie_id` | MovieLens movie ID, 1–1682 |
| `title` | `string` |   | `u.item.raw_title` (parsed) | Clean title, e.g. `"Toy Story"` — year stripped from `"Toy Story (1995)"` at load time |
| `year` | `int` (nullable) |   | `u.item.raw_title` (parsed) | Release year extracted from the parenthesized suffix |
| `tmdbId` | `int` (nullable) |   | TMDB match step (T-07) | Cross-reference to TMDB; null for unmatched movies (3 known TV miniseries) |

- **Expected cardinality:** ~1,680 nodes (1,682 MovieLens titles minus any dropped at load — finalized in T-10)
- **Notes:** `title` is parsed (year stripped) so the full-text index returns clean matches. `year` is kept as a separate property for filter queries. `tmdbId` is the bridge to TMDB enrichment — queries needing poster URLs, etc., join through this ID at app time, not via stored properties.

### 4.3 `:Genre`

Categorical reference node for movie genres.

| Property | Type | Key? | Source | Description |
|---|---|---|---|---|
| `name` | `string` | ✅ unique | MovieLens genre vocabulary | One of 19 genres, e.g. `"Action"`, `"Drama"` |

- **Expected cardinality:** 19 nodes
- **Notes:** Using `name` (not a numeric ID) as the key is intentional — readability in Cypher (`MATCH (:Genre {name: "Comedy"})` reads naturally) and there are only 19 of them so a string PK has negligible cost. Genre vocabulary is closed for MovieLens 100K.

### 4.4 `:Director`

Person node for film directors.

| Property | Type | Key? | Source | Description |
|---|---|---|---|---|
| `name` | `string` | ✅ unique | TMDB `crew.name` where `job = "Director"` | Full name, e.g. `"Steven Spielberg"` |

- **Expected cardinality:** ~1,500 nodes
- **Notes:** Using `name` as the key is a deliberate trade-off — lossy if two real directors share a name (homonym collision), but TMDB person IDs are not exposed in MovieLens and the readability/teaching gain outweighs the rare collision risk in this corpus. If collisions become a problem later, migrate to `tmdbPersonId`.

### 4.5 `:Actor`

Person node for film actors.

| Property | Type | Key? | Source | Description |
|---|---|---|---|---|
| `name` | `string` | ✅ unique | TMDB `cast.name`, top 10 by `order` per movie | Full name, e.g. `"Tom Hanks"` |

- **Expected cardinality:** ~6,000 nodes (deduplicated across movies)
- **Notes:** Same homonym caveat as `:Director`. Capping at 10 actors per movie (by TMDB billing order) keeps the graph dense in lead-actor signal while preventing actor-edge explosion from movies with 50+ credited cast.

### 4.6 `:Occupation`

Categorical reference node for the user's self-reported job.

| Property | Type | Key? | Source | Description |
|---|---|---|---|---|
| `name` | `string` | ✅ unique | MovieLens occupation vocabulary | One of 21 occupations, e.g. `"student"`, `"engineer"`, `"writer"` |

- **Expected cardinality:** 21 nodes
- **Notes:** Modeled as a node (not a `:User` property) specifically so CQ-1.5 ("top genres among users sharing user U's occupation") and CQ-2.4 (cold-start by demographics) can pivot through it in one hop.

### 4.7 `:Gender`

Categorical reference node for the user's self-reported gender.

| Property | Type | Key? | Source | Description |
|---|---|---|---|---|
| `name` | `string` | ✅ unique | MovieLens `u.user.gender` | `"M"` or `"F"` |

- **Expected cardinality:** 2 nodes
- **Notes:** **Lopsided supernode** — with only 2 nodes, every gender edge fans out to ~470 users. Excluded from explanation paths (see §7) but kept as a queryable node so cold-start logic (CQ-2.4) can compute popularity among same-gender users.

---

## 5. Relationship Specifications

Per relationship type: pattern, properties, expected cardinality, and design notes.

### 5.1 `:RATED`

The core signal of the entire RecSys — a user's rating of a movie.

- **Pattern:** `(:User)-[:RATED]->(:Movie)`

| Property | Type | Source | Description |
|---|---|---|---|
| `rating` | `float` | `u.data.rating` | 1.0–5.0 (MovieLens uses integers; stored as float for arithmetic consistency) |
| `timestamp` | `int` | `u.data.timestamp` | Unix epoch seconds; supports future "recent ratings" filters |

- **Expected cardinality:** 100,000 edges total; ~106 per user on average (range 20–737)
- **Notes:** Properties live on the *edge*, not on the user or movie node, because the rating is the pair-specific fact — the *only* datum that exists only because this user and this movie meet. This is the textbook teaching example of why edges in a property graph carry properties (see §7 for the SQL vs Cypher comparison).

### 5.2 `:IN_GENRE`

Membership of a movie in a genre.

- **Pattern:** `(:Movie)-[:IN_GENRE]->(:Genre)`
- **Properties:** none

- **Expected cardinality:** ~3,000 edges (avg ~1.8 genres per movie; min 1, max 6)
- **Notes:** A movie can belong to multiple genres (many-to-many). In SQL this would be a junction table; in the graph it's the natural relationship. No properties — a movie either is in a genre or isn't.

### 5.3 `:DIRECTED_BY`

Movie's directorial credit.

- **Pattern:** `(:Movie)-[:DIRECTED_BY]->(:Director)`
- **Properties:** none

- **Expected cardinality:** ~1,800 edges (most movies have 1 director; co-directed movies have 2+)
- **Notes:** Co-directors are modeled as multiple edges, not as a list-property on the movie. Lets CQ-1.4 ("directors U rates highest") and CQ-2.2 ("movies sharing a director") work uniformly with no special-casing.

### 5.4 `:FEATURES_ACTOR`

Movie's acting credit, capped at the top 10 by billing order.

- **Pattern:** `(:Movie)-[:FEATURES_ACTOR]->(:Actor)`

| Property | Type | Source | Description |
|---|---|---|---|
| `billing_order` | `int` | TMDB `cast.order` + 1 | 1 = lead, 2 = second-billed, … 10 = tenth-billed |

- **Expected cardinality:** ~15,000 edges (avg ~9 per movie, capped at 10)
- **Notes:** `billing_order` lives on the *edge* because it's pair-specific — Tom Hanks's billing differs across his movies. Cap at 10 prevents actor-edge explosion from movies with 50+ credited cast; the top of the bill carries the audience-recognizable signal. TMDB's native `order` is 0-indexed; we store as 1-indexed for readability.

### 5.5 `:HAS_OCCUPATION`

User's self-reported occupation.

- **Pattern:** `(:User)-[:HAS_OCCUPATION]->(:Occupation)`
- **Properties:** none

- **Expected cardinality:** 943 edges (exactly one per user — every MovieLens 100K user has exactly one occupation)
- **Notes:** Modeled as a relationship (not a `:User` property) so CQ-1.5 and CQ-2.4 can pivot through it: `MATCH (u)-[:HAS_OCCUPATION]->(o)<-[:HAS_OCCUPATION]-(peer)` gives all same-occupation peers in one hop.

### 5.6 `:HAS_GENDER`

User's self-reported gender.

- **Pattern:** `(:User)-[:HAS_GENDER]->(:Gender)`
- **Properties:** none

- **Expected cardinality:** 943 edges (exactly one per user)
- **Notes:** Same motivation as `:HAS_OCCUPATION` — modeled as a relationship so cold-start demographic queries (CQ-2.4) can traverse it. **Excluded from explanation paths** because of the supernode shape (see §4.7 and §7).

---

## 6. Constraints & Indexes

The schema requires **7 uniqueness constraints** (one per node label) and **1 full-text index** (on `Movie.title`). All are created **before** data load — the constraints prevent duplicate nodes during `MERGE`, and the full-text index supports CQ-4.5 (movie search by partial title).

### Single block — paste into Neo4j Browser

```cypher
// --- Uniqueness constraints (one per node label) ---
CREATE CONSTRAINT user_id IF NOT EXISTS
  FOR (u:User) REQUIRE u.userId IS UNIQUE;

CREATE CONSTRAINT movie_id IF NOT EXISTS
  FOR (m:Movie) REQUIRE m.movieId IS UNIQUE;

CREATE CONSTRAINT genre_name IF NOT EXISTS
  FOR (g:Genre) REQUIRE g.name IS UNIQUE;

CREATE CONSTRAINT director_name IF NOT EXISTS
  FOR (d:Director) REQUIRE d.name IS UNIQUE;

CREATE CONSTRAINT actor_name IF NOT EXISTS
  FOR (a:Actor) REQUIRE a.name IS UNIQUE;

CREATE CONSTRAINT occupation_name IF NOT EXISTS
  FOR (o:Occupation) REQUIRE o.name IS UNIQUE;

CREATE CONSTRAINT gender_name IF NOT EXISTS
  FOR (g:Gender) REQUIRE g.name IS UNIQUE;

// --- Full-text index for movie title search (CQ-4.5) ---
CREATE FULLTEXT INDEX movie_title_search IF NOT EXISTS
  FOR (m:Movie) ON EACH [m.title];
```

### Why these constraints

| Constraint | Why |
|---|---|
| `user_id`, `movie_id` | Numeric primary keys from MovieLens — unique by definition; `MERGE` on these is the load-idempotency anchor. |
| `genre_name`, `occupation_name`, `gender_name` | Closed vocabularies — `MERGE` on `name` must idempotently target the one canonical node. |
| `director_name`, `actor_name` | Despite the homonym caveat (see §4.4), `MERGE` still needs a unique key so repeated loads don't create duplicate "Steven Spielberg" nodes. |

### Why this index

| Index | Why |
|---|---|
| `movie_title_search` (full-text on `Movie.title`) | CQ-4.5 requires partial-title matching. A B-tree index would only help equality/prefix queries; full-text handles substring and tokenized matches (`"toy"` → `"Toy Story"`, `"Toy Story 2"`). |

### What we deliberately do NOT create

- **No range/B-tree index on `User.age`** — only 943 users, a full scan is trivial.
- **No index on `RATED.rating` or `RATED.timestamp`** — recommendation queries traverse via relationship patterns, not edge-property lookups; relationship indexes would just take up the AuraDB Free budget.
- **No existence constraints** (`REQUIRE … IS NOT NULL`) — that's a Neo4j Enterprise feature. AuraDB Free is Community-tier. We rely on the T-10 loader to enforce non-null at write time.

### When these run

The constraint/index block is the **first** statement executed in T-10 (`04_graph_load.ipynb`), before any `MERGE`. `IF NOT EXISTS` makes the block idempotent: re-running on a populated database is a no-op, so the load notebook can safely be re-executed.

---

## 7. Query Conventions

Rules that apply across every Cypher query in the project. These keep results consistent, prevent path explosion, and make explanations interpretable.

### 7.1 Variable-length paths — undirected, capped at 4 hops

All multi-hop explanation queries use the pattern:

```cypher
MATCH path = (u:User {userId: $uid})-[*1..4]-(m:Movie {movieId: $mid})
```

- **Undirected** (`-[*1..4]-`, not `-[*1..4]->`) — explanations don't care about stored edge direction. A user finding a path *through* a director should not be blocked by the fact that `:DIRECTED_BY` happens to point from movie to director.
- **Hop cap of 4** — empirically covers the meaningful explanation patterns:
  - **2 hops:** `User → Movie → Genre` (rare; the recommended movie itself shares a genre with one U rated)
  - **3 hops:** `User → Movie → Director → Movie` (common — the "Nolan style" explanation)
  - **4 hops:** `User → Movie → Actor → Movie → Director` (composite — actor + director together)
- **5+ hops** explode combinatorially (a 5-hop traversal through `:RATED` alone can touch hundreds of thousands of paths) and rarely add interpretable explanation power.

### 7.2 Result caps

| Query class | Cap | Reason |
|---|---|---|
| Explanation paths per recommendation | 3 | UI shows up to 3 alternative reasons; more is clutter, not insight. |
| Recommendations per strategy | 10 | One screen of cards. |
| Neighborhood nodes for pyvis (Graph Explorer) | 50 | pyvis becomes unreadable beyond ~50 nodes; rendering also slows noticeably. |
| Search-by-title results (CQ-4.5) | 25 | Dropdown limit in the explorer search box. |

### 7.3 Gender exclusion in explanation paths

`:Gender` has only 2 nodes (`M`, `F`), making every gender edge a supernode hub. A path through it is uninformative ("recommended because you and 470 other users are both female"). Every explanation query explicitly excludes it:

```cypher
MATCH path = (u:User)-[*1..4]-(m:Movie)
WHERE NOT ANY(n IN nodes(path) WHERE n:Gender)
```

`:Occupation` (21 nodes) is a partial supernode — kept in paths but contributes less weight in ranking, because the *frequency* of a shared-occupation path type is less interpretive than the relevance of, say, a shared-director path.

### 7.4 `NOT EXISTS` guard for unseen movies

Every recommendation query must exclude movies the target user has already rated:

```cypher
MATCH (u:User {userId: $uid})
…  // build candidate movies (e.g. via genre, director, collaborative pattern)
WHERE NOT EXISTS {
  MATCH (u)-[:RATED]->(candidate)
}
```

This is the graph-native equivalent of the SQL `LEFT JOIN … WHERE rating IS NULL` anti-join pattern.

### 7.5 SQL vs Cypher — the teaching moment

CQ-2.1 in plain English: *"Recommend movies in user 42's favorite genres that they have not yet rated, ranked by the user's average rating in those genres."*

**In SQL** (assuming a normalized schema with `movie_genres` and `genres` tables):

```sql
SELECT m_rec.title,
       g.name AS genre,
       AVG(r1.rating) AS user_genre_score
FROM   users u
JOIN   ratings        r1     ON r1.user_id  = u.user_id  AND r1.rating >= 4
JOIN   movie_genres   mg1    ON mg1.movie_id = r1.movie_id
JOIN   genres         g      ON g.genre_id   = mg1.genre_id
JOIN   movie_genres   mg2    ON mg2.genre_id = g.genre_id
JOIN   movies         m_rec  ON m_rec.movie_id = mg2.movie_id
LEFT JOIN ratings     r_seen ON r_seen.user_id  = u.user_id
                            AND r_seen.movie_id = m_rec.movie_id
WHERE  u.user_id = 42
  AND  r_seen.rating IS NULL          -- anti-join: user hasn't rated m_rec
GROUP BY m_rec.movie_id, m_rec.title, g.name
ORDER BY user_genre_score DESC
LIMIT 10;
```

Six tables, seven joins (one anti-join), an explicit GROUP BY across two keys. The `movie_genres` junction table appears **twice** — once to find the user's liked genres, once to find candidate movies sharing those genres. If either junction reference is forgotten, the query silently returns wrong results.

**In Cypher:**

```cypher
MATCH (u:User {userId: 42})-[r1:RATED]->(:Movie)-[:IN_GENRE]->(g:Genre)
                                                       <-[:IN_GENRE]-(rec:Movie)
WHERE r1.rating >= 4
  AND NOT EXISTS { MATCH (u)-[:RATED]->(rec) }
WITH  rec, g, AVG(r1.rating) AS user_genre_score
RETURN rec.title, g.name AS genre, user_genre_score
ORDER BY user_genre_score DESC
LIMIT 10;
```

Same answer. **One traversal pattern.** The graph structure `(Movie)-[:IN_GENRE]->(Genre)<-[:IN_GENRE]-(Movie)` replaces both `movie_genres` references — junction tables are not a thing the modeler has to maintain; they are implicit in the relationship type.

This is **the teaching moment** Data Analysts who already know SQL connect with first: a JOIN-heavy query collapses to a single pattern as soon as the data is in graph form.

### 7.6 The notebook is documentation, not execution

`03_graph_schema.ipynb` is a **design document**. It contains no Neo4j connection, no `session.run`, no live queries. All Cypher shown is documentation — copy into Neo4j Browser if you want to try it. Loading happens in `04_graph_load.ipynb` (T-10); recommendation queries live in `src/recommender.py` (T-13).

---

## 8. Design Decisions Log

Every non-trivial schema choice and the alternative we rejected. Reading this top-to-bottom is the fastest way to understand *why* the schema looks the way it does. Each entry is small, but together they are the rationale a reviewer (or a student doing the course) should expect.

### How to read this table

- **Decision** — what we settled on, in one phrase.
- **Considered** — the realistic alternative(s) we evaluated.
- **Why** — the deciding reason, anchored to a CQ, a constraint, or a teaching goal.

| # | Decision | Considered | Why |
|---|---|---|---|
| D1 | **Labeled Property Graph (Neo4j LPG)**, not RDF/OWL | RDF triplestore + SPARQL | LPG models edge properties (like `RATED.rating`, `FEATURES_ACTOR.billing_order`) natively. In RDF the same fact requires reification — a teaching-grade dataset would spend half the lesson explaining the workaround. |
| D2 | **`name` as the key for `:Genre`, `:Director`, `:Actor`, `:Occupation`, `:Gender`** | Synthetic numeric ID + `name` property | Readability in Cypher: `MATCH (:Director {name: "Nolan"})` is self-explanatory in a demo. Trade-off: homonym collisions (two real directors named "John Smith") collapse to one node. Acceptable for MovieLens 100K; migration path documented in §10. |
| D3 | **`:Occupation` and `:Gender` modeled as nodes**, not as `:User` properties | Store `occupation`, `gender` as string properties on `:User` | CQ-1.5 ("top genres for users sharing U's occupation") and CQ-2.4 (cold-start by demographics) need to *pivot* through these fields in a single hop. Property-based filtering would require post-hoc grouping; node-based modeling makes the pivot a 2-hop traversal. |
| D4 | **`:Gender` kept as a node despite the supernode shape** | Drop `:Gender` entirely; store only as a `:User` property | Cold-start (CQ-2.4) genuinely needs the demographic pivot. The supernode cost is mitigated by *excluding `:Gender` from explanation paths in §7.3* — kept in the graph, excluded from traversal. Two-track usage. |
| D5 | **`User.age` stored as raw int**, not bucketed | Store age bucket (`"18-24"`, `"25-34"`, …) as a `:User` property or as edges to `:AgeBucket` nodes | Buckets are a tunable hyperparameter of cold-start. Hard-coding them in the schema would freeze the experiment. Keeping the raw value lets the cold-start strategy in `src/recommender.py` choose its own window (e.g. ±5 years) at query time. Bucketing revisited in §10. |
| D6 | **`rating`, `timestamp` live on the `:RATED` edge** | Store as properties on `:User` or `:Movie`, or as a separate `:Rating` reified node | The rating is *pair-specific* — it exists only because this user met this movie. It cannot belong to either node alone. Reifying as a `:Rating` node would 5× the relationship count (100k → 500k+) and make the simplest CQ (CQ-1.1) a 2-hop pattern. The edge-property model is canonical. |
| D7 | **`billing_order` on the `:FEATURES_ACTOR` edge** | Store as an `Actor` property; or store an ordered list on `:Movie`; or use a `:Role` node | Same pair-specific argument as D6. Tom Hanks is billed differently in different films. Lists would not be queryable in Cypher with the same pattern matching. The edge-property approach matches the canonical "two-mode network with weighted edges" pattern. |
| D8 | **Cap actors at top 10 by TMDB billing order** | Load all credited cast (often 50+ per movie) | (1) Keeps `:FEATURES_ACTOR` at ~15k edges instead of ~70k+. (2) Lead-role signal is denser at the top of the bill — a 50th-billed extra contributes noise, not signal. (3) Comfortably under the AuraDB Free ~200k edge ceiling. Cap value (10) is a §10 open question. |
| D9 | **No materialized `:SIMILAR_TO` or `:COLLABORATED_WITH` edges** — compute live | Pre-compute and store these as derived edges at load time | (1) MovieLens 100K is small enough that live traversal is sub-second. (2) Materializing actor-actor collaboration would explode to O(actors²) edges — quickly above the AuraDB Free ceiling. (3) **Teaching moment:** the whole point of Cypher's multi-hop pattern is that you *don't* pre-compute these. See §10 for the threshold at which materialization becomes a real option. |
| D10 | **No existence constraints (`REQUIRE … IS NOT NULL`)** | Add `REQUIRE u.userId IS NOT NULL` etc. | Existence constraints are Neo4j Enterprise-only. AuraDB Free is Community-tier; the statement would error at load. Non-null is enforced upstream in the T-10 loader (Python-side validation before `MERGE`). |
| D11 | **`Movie.title` parsed (year stripped) at load time** | Store the MovieLens raw title `"Toy Story (1995)"` verbatim | (1) The full-text index `movie_title_search` returns clean matches without the year noise. (2) UI rendering doesn't need to strip-the-year on every display. (3) `year` lives in its own typed property — filterable and sortable. Parsing is a one-line regex at load. |
| D12 | **`Movie.year` as a separate `int` property** | Encode year only inside the title string | Filter queries ("movies from the 90s") become a property comparison instead of a substring parse. Costs one extra column on a 1,680-row node set — negligible. |
| D13 | **`Movie.tmdbId` stored as a property, no other TMDB metadata in the graph** | Pull poster URLs, runtime, overview text into the graph as movie properties | The graph is for *recommendation traversal*, not as a movie catalog. The app pulls extra TMDB metadata at render time via the TMDB API or a cached lookup keyed on `tmdbId`. Keeps the graph focused on the relationships RecSys uses; keeps node payloads small. |
| D14 | **Undirected traversal in multi-hop queries** (`-[*1..4]-`, not `-[*1..4]->`) | Direction-respecting traversal | The stored direction of `:DIRECTED_BY` (movie → director) is a load-time convention. An explanation path from a user to a recommended movie does not care about that convention — it cares that the two are connected through Nolan. Undirected matching makes that natural. See §7.1. |
| D15 | **Hop cap of 4 in variable-length paths** | Allow `*1..6` or unbounded | 5+ hops via `:RATED` explodes combinatorially (a 5-hop traversal from one user can touch hundreds of thousands of paths). Empirically all meaningful explanations live at 2–4 hops. Hard cap protects AuraDB Free's per-query memory budget. |
| D16 | **`:Gender` excluded from explanation paths** (query-level, not schema-level) | Drop the node from the graph entirely | The information is needed for cold-start (CQ-2.4) but not for explanations. Excluding in queries (via `WHERE NOT ANY(n IN nodes(path) WHERE n:Gender)`) keeps both use cases possible from the same data. See §7.3. |
| D17 | **7-label / 6-relationship surface area is final** | Add `:Country`, `:Language`, `:Studio`, `:Keyword`, etc. from TMDB | None of the 19 CQs justify those labels. Per the design criteria in §1, no CQ → not in the graph. Revisited in §10 if a future CQ legitimately needs them. |

Decisions D2, D4, D6, D7, D9, D14, D15 are the ones a student is most likely to ask "why?" about. Each of them deliberately *gives something up* — they are the ones where having the recorded rationale matters most.

---

## 9. Data Source Mapping

For every node label, every relationship type, and every property: which source file column it comes from and any transformation applied at load time. This is the contract between T-08 (this design doc) and T-10 (the loader notebook).

### Source files

| Source | What it is | Used for |
|---|---|---|
| `data/raw/ml-100k/u.user` | MovieLens user demographics (TSV, 943 rows) | `:User`, `:Occupation`, `:Gender` nodes; `:HAS_OCCUPATION`, `:HAS_GENDER` edges |
| `data/raw/ml-100k/u.item` | MovieLens movie metadata (pipe-delim, 1,682 rows) | `:Movie` nodes; `:Genre` nodes (via genre flags); `:IN_GENRE` edges |
| `data/raw/ml-100k/u.data` | MovieLens ratings (TSV, 100,000 rows) | `:RATED` edges |
| `data/raw/ml-100k/u.genre` | MovieLens genre vocabulary (19 rows) | `:Genre` node names |
| `data/raw/ml-100k/u.occupation` | MovieLens occupation vocabulary (21 rows) | `:Occupation` node names |
| `data/enriched/enriched_movies.csv` | TMDB enrichment output from T-07 (1,679 of 1,682 matched) | `:Director`, `:Actor` nodes; `:DIRECTED_BY`, `:FEATURES_ACTOR` edges; `Movie.tmdbId` |

> The enriched CSV serializes nested fields (`cast`, `directors`, `genre_names`) as JSON strings inside CSV cells. The T-10 loader JSON-parses these columns on read. See [`docs/02_enrichment_approach.md`](../docs/02_enrichment_approach.md) for the pipeline that produced this file.

### 9.1 Node property mapping

#### `:User` (943 rows)

| Property | Source column | Transformation |
|---|---|---|
| `userId` | `u.user.user_id` | Cast to `int`; unique key for `MERGE` |
| `age` | `u.user.age` | Cast to `int`; no bucketing (see D5) |

> `u.user` also contains `gender` and `occupation` — these are NOT stored on `:User`. They flow into `:HAS_GENDER` and `:HAS_OCCUPATION` edges instead (see 9.2).

#### `:Movie` (~1,680 rows after TMDB match)

| Property | Source column | Transformation |
|---|---|---|
| `movieId` | `u.item.movie_id` | Cast to `int`; unique key for `MERGE` |
| `title` | `u.item.raw_title` | Regex strip of trailing ` (YYYY)`; result is the clean title (see D11) |
| `year` | `u.item.raw_title` | Regex extract of the trailing 4-digit year; `null` if absent |
| `tmdbId` | `enriched_movies.tmdb_id` | Joined on `movieId`; `null` for ~3 movies that did not match in T-07 |

#### `:Genre` (19 rows)

| Property | Source column | Transformation |
|---|---|---|
| `name` | `u.genre.genre_name` | Verbatim string; e.g. `"Action"`, `"Drama"`, `"Film-Noir"` |

#### `:Director` (~1,500 rows)

| Property | Source column | Transformation |
|---|---|---|
| `name` | `enriched_movies.directors` (JSON list of names per movie) | Already filtered to directors-only upstream in T-07. JSON-parse the column, iterate names, dedup via `MERGE` |

#### `:Actor` (~6,000 rows)

| Property | Source column | Transformation |
|---|---|---|
| `name` | `enriched_movies.cast` (JSON list of cast objects per movie) | `cast` is already top-10-by-billing per movie (T-07). JSON-parse the column, extract each `name`, dedup via `MERGE` |

#### `:Occupation` (21 rows)

| Property | Source column | Transformation |
|---|---|---|
| `name` | `u.occupation.occupation_name` | Verbatim string; e.g. `"engineer"`, `"student"`, `"writer"` |

#### `:Gender` (2 rows)

| Property | Source column | Transformation |
|---|---|---|
| `name` | `u.user.gender` (distinct) | Verbatim single-char: `"M"` or `"F"` |

### 9.2 Relationship property mapping

#### `:RATED` (100,000 edges)

| Endpoint | Source |
|---|---|
| `:User`  | `u.data.user_id` → `User.userId` |
| `:Movie` | `u.data.movie_id` → `Movie.movieId` |

| Property | Source column | Transformation |
|---|---|---|
| `rating` | `u.data.rating` | Cast `int` → `float` for arithmetic consistency (see D6) |
| `timestamp` | `u.data.timestamp` | Cast to `int`; Unix epoch seconds, preserved as-is |

#### `:IN_GENRE` (~3,000 edges)

| Endpoint | Source |
|---|---|
| `:Movie` | `u.item.movie_id` → `Movie.movieId` |
| `:Genre` | `u.item.genre_flags` (19 boolean columns, one per genre) → unpivoted to one row per True flag |

No edge properties.

#### `:DIRECTED_BY` (~1,800 edges)

| Endpoint | Source |
|---|---|
| `:Movie` | `enriched_movies.movieId` → `Movie.movieId` |
| `:Director` | `enriched_movies.directors` (JSON list) → one edge per director name → `Director.name` |

No edge properties. Co-directed movies yield multiple edges from the same `:Movie` (see §5.3).

#### `:FEATURES_ACTOR` (~15,000 edges)

| Endpoint | Source |
|---|---|
| `:Movie` | `enriched_movies.movieId` → `Movie.movieId` |
| `:Actor` | `enriched_movies.cast` (JSON list of top-10 cast objects) → one edge per cast entry → `Actor.name` |

| Property | Source column | Transformation |
|---|---|---|
| `billing_order` | position in `enriched_movies.cast` list (0-indexed) — or the per-cast-object `order` field if present | Add 1 so 1 = lead, 10 = tenth-billed. `cast` is already filtered to top 10 by T-07 (see D8), so no further filtering needed at load time. |

#### `:HAS_OCCUPATION` (943 edges)

| Endpoint | Source |
|---|---|
| `:User` | `u.user.user_id` → `User.userId` |
| `:Occupation` | `u.user.occupation` → `Occupation.name` |

No edge properties.

#### `:HAS_GENDER` (943 edges)

| Endpoint | Source |
|---|---|
| `:User` | `u.user.user_id` → `User.userId` |
| `:Gender` | `u.user.gender` → `Gender.name` |

No edge properties.

### 9.3 Transformations done in T-10 (the loader), not here

The mappings above describe **what** the loader does. The actual code in `04_graph_load.ipynb` (T-10) handles:

- Reading MovieLens files with the correct delimiter and encoding (`u.item` is Latin-1, not UTF-8)
- The `raw_title` → `(title, year)` regex parse
- Unpivoting `u.item`'s 19 boolean genre columns into one row per `(movieId, genre)` pair
- JSON-parsing the `cast`, `directors`, and `genre_names` columns of `enriched_movies.csv`
- Adding 1 to the cast position to get 1-indexed `billing_order`
- Idempotent `MERGE` for every node and relationship, so re-running the loader does not duplicate

> Top-10 cast filtering happens **upstream** in T-07 (not T-10) — the loader trusts `enriched_movies.cast` to already contain at most 10 entries per movie in billing order.

If a mapping in this section is changed, the loader must change in lockstep — and the constraint block in §6 must still hold afterwards.

---

## 10. Open Questions / Future Work

Items intentionally deferred. Each entry records what we did **not** decide, what would trigger re-opening the design, and which task or section to return to when re-opening.

### OQ-1 — Age bucketing for cold-start

- **What we did:** Stored `User.age` as a raw `int` (see D5).
- **What we did not decide:** The age window used by the cold-start strategy. Candidates: ±5 years; bucketed groups (`18-24, 25-34, 35-49, 50+`); decade buckets (`20s, 30s, …`); Gaussian-weighted window.
- **Trigger to revisit:** When T-13 (`src/recommender.py`) implements `cold_start_demographic()`. The choice will likely be made empirically by inspecting the recommendation quality on a held-out cold-start cohort.
- **Schema impact if reopened:** None expected — the experiment runs entirely query-side. Schema would only change if we decide to *materialize* a bucket node, which is unlikely.

### OQ-2 — Unmatched movies after TMDB enrichment

- **What we did:** Allowed `Movie.tmdbId` to be nullable (see §4.2). T-07 left ~3 MovieLens titles unmatched (suspected TV miniseries with ambiguous titles).
- **What we did not decide:** Whether to (a) drop unmatched movies from the graph, (b) keep them with `tmdbId = null` and accept they will have no `:DIRECTED_BY` / `:FEATURES_ACTOR` edges, or (c) manually match them.
- **Trigger to revisit:** During T-10 load — once the unmatched list is final and small, manual matching is the cheapest fix. If the list grows in a later TMDB refresh, re-evaluate.
- **Schema impact if reopened:** None — option (b) is the current default and already supported.

### OQ-3 — Materializing derived edges (`:SIMILAR_TO`, `:COLLABORATED_WITH`)

- **What we did:** Per D9, do not materialize. Compute live at query time.
- **What we did not decide:** The performance threshold at which live computation becomes too slow. A rough rule of thumb: if the AuraDB Free instance routinely exceeds ~500 ms on the composite multi-signal query, pre-computing `:SIMILAR_TO` between movies (top-K neighbors per movie) becomes attractive.
- **Trigger to revisit:** If end-to-end recommendation latency on the Streamlit app exceeds 1 s in the demo. If the dataset is later scaled past MovieLens 1M, materialization is almost certainly needed.
- **Schema impact if reopened:** Adds `(:Movie)-[:SIMILAR_TO {score, computed_at}]->(:Movie)` and/or `(:Actor)-[:COLLABORATED_WITH {count}]->(:Actor)`. Both are listed in `CLAUDE.md` as "derived" relationship types already, meaning the surface area is pre-blessed but the load step is deferred.

### OQ-4 — Homonym handling for `:Director` / `:Actor`

- **What we did:** Used `name` as the key (see D2). Collisions collapse two real people into one node.
- **What we did not decide:** Whether to migrate to `tmdbPersonId` as the key. TMDB credits do expose person IDs in the raw payload — we just chose not to store them.
- **Trigger to revisit:** If a teaching scenario or an explanation path is observed to be wrong because of a collision (e.g., recommendations attributed to the wrong John Smith).
- **Schema impact if reopened:** Change of primary key from `name` to `tmdbPersonId` on `:Director` and `:Actor`; `name` becomes a non-unique property. Loader must update `MERGE` clauses. The constraint block in §6 changes for those two labels.

### OQ-5 — Actor cap value (currently 10)

- **What we did:** Capped `:FEATURES_ACTOR` at top 10 by TMDB billing order (see D8).
- **What we did not decide:** Whether 10 is the right number. 5 is more "lead-role only"; 20 covers richer ensemble casts; the cap could also vary per genre (e.g., 5 for action, 15 for ensemble drama).
- **Trigger to revisit:** If CQ-2.2 ("movies sharing an actor with U's top-rated") underperforms in user-testing — too few hits means the cap is too tight; too noisy means it is too loose.
- **Schema impact if reopened:** None — change is in the loader, not the schema. Just a different filter constant in T-10.

### OQ-6 — Recency / time-decay in `RATED.timestamp`

- **What we did:** Stored `timestamp` on `:RATED` (see 5.1) but no recommendation strategy currently uses it.
- **What we did not decide:** Whether to weight recent ratings more heavily, or to exclude ratings older than some cutoff. MovieLens 100K spans 1997–1998, so the temporal range is narrow; the question becomes more interesting on larger datasets.
- **Trigger to revisit:** Optional T-13 extension: "Recent-taste recommender" — a fifth strategy that decays `r1.rating` by `exp(-(now - r1.timestamp) / τ)`.
- **Schema impact if reopened:** None — `timestamp` is already on the edge for exactly this use case.

### OQ-7 — TMDB metadata expansion beyond credits

- **What we did:** Stored only `tmdbId` on `:Movie` (see D13). Posters, runtime, overview, etc. are pulled at app-render time, not stored in the graph.
- **What we did not decide:** Whether expanding to add `:Studio`, `:Country`, `:Language`, or `:Keyword` nodes would unlock interesting CQs. Currently no CQ asks for them — see D17.
- **Trigger to revisit:** A new CQ from the course iteration that legitimately needs one of these labels (e.g., "recommend foreign films a user has not seen" → needs `:Language`).
- **Schema impact if reopened:** Adds new node labels and a corresponding `IN_*` relationship. Stay within AuraDB Free's ~200k ceiling — the inventory budget table in §3 has headroom.

### OQ-8 — pyvis rendering ceiling for Graph Explorer

- **What we did:** Capped neighborhood views at 50 nodes (see §7.2).
- **What we did not decide:** Whether to switch to a different visualization (vis-network, Cytoscape) for larger neighborhood views, or to add server-side aggregation ("collapse all actors of this movie into one summary node") for high-degree nodes.
- **Trigger to revisit:** If course feedback indicates 50 is too restrictive for the Graph Explorer teaching moment.
- **Schema impact if reopened:** None — purely a presentation choice.

### Items deliberately NOT on this list

- Dropping/renaming an existing node label or relationship type — the schema surface area is final per D17.
- Switching graph databases (e.g., to ArangoDB, AWS Neptune, TigerGraph) — out of scope per §1.
- Streaming / incremental graph updates — out of scope per §1.

When any open question is closed, update D5–D17 in §8 with the new decision and add a row to this section noting when and why it was closed.